In [15]:
import requests
from bs4 import BeautifulSoup
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
from datetime import date, timedelta
import pandas as pd

# Download the VADER lexicon if you haven't already
try:
    sid = SentimentIntensityAnalyzer()
except LookupError:
    nltk.download('vader_lexicon')
    sid = SentimentIntensityAnalyzer()

def get_news_sentiment(companyname, num_days=3):
    """
    Fetches news headlines for a company, analyzes sentiment, and returns an overall score.

    Args:
        companyname (str): The name of the company to search for.
        num_days (int): The number of past days to consider.

    Returns:
        dict: A dictionary containing the overall sentiment score, positive news, negative news, and neutral news.
    """

    today = date.today()
    news = {"positive": [], "negative": [], "neutral": []}
    overall_score = 0
    total_articles = 0

    for i in range(num_days):
        search_date = today - timedelta(days=i)
        date_string = search_date.strftime("%Y-%m-%d")
        query = f"{companyname} news after:{date_string}"  # Date-restricted query

        url = f"https://www.google.com/search?q={query}&tbm=nws"
        headers = {"User-Agent": "Mozilla/5.0"}  # Important to avoid bot detection

        try:
            response = requests.get(url, headers=headers)
            response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
            soup = BeautifulSoup(response.content, "html.parser")
            attrs = {'class': 'Gx5Zad fP1Qef xpd EtOod pkphOe',
            'jscontroller': "something",
            'jsaction': "mousedown",
            'data-sok': "true"}
            for g in soup.find_all('div', attrs=attrs):

                try:
                    title = g.find('div', class_='mCBkyc y355M JQe2Ld gsrt KMQW0d kno-fb-ctx').text
                    link = g.find('a', class_='VDXfz')['href']
                    total_articles += 1

                    sentiment_scores = sid.polarity_scores(title)
                    compound_score = sentiment_scores['compound']

                    if compound_score >= 0.05:
                        news["positive"].append({"title": title, "link": link})
                        overall_score += compound_score
                    elif compound_score <= -0.05:
                        news["negative"].append({"title": title, "link": link})
                        overall_score += compound_score
                    else:
                        news["neutral"].append({"title": title, "link": link})

                except Exception as e:
                    print(f"Error processing headline: {e}")
                    continue
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            continue

    if total_articles > 0:
        overall_score /= total_articles
    return {
        "overall_score": overall_score,
        "positive_news": news["positive"],
        "negative_news": news["negative"],
        "neutral_news": news["neutral"],
        "total_articles": total_articles
    }

def analyze_companies_from_file(filename):
    """
    Reads company names and index names from a CSV file and performs sentiment analysis for each company.

    Args:
        filename (str): The path to the CSV file containing company names and index names.
                         Assumes the CSV has columns named 'companyname' and 'indexname'.
    """
    try:
        df = pd.read_csv(filename)
        # Extract company names and index names from the CSV
        companynames = df['companyname'].tolist()
        indexnames = df['indexname'].tolist()  # Assuming you have a column named 'indexname'
    except FileNotFoundError:
        print(f"Error: File not found: {filename}")
        return
    except KeyError as e:
        print(f"Error: Column not found in {filename}: {e}")
        return
    except Exception as e:
        print(f"Error reading file: {e}")
        return

    for company, index in zip(companynames, indexnames):
        results = get_news_sentiment(company)
        print(f"Sentiment Analysis for {company} ({index}):")
        print(f"  Overall Score: {results['overall_score']:.2f}")
        print(f"  Positive News Count: {len(results['positive_news'])}")
        print(f"  Negative News Count: {len(results['negative_news'])}")
        print(f"  Neutral News Count: {len(results['neutral_news'])}")
        print("-" * 40)

# Example Usage (assuming you have a CSV file):
file_path = 'C://Users//manoj//Downloads//Major project data//Major pro source codes//DATASETS//filtered_indices_output.csv'  # Replace with the actual path to your CSV file
analyze_companies_from_file(file_path)


Sentiment Analysis for Bajaj Auto Limited (BAJAJ-AUTO.NS):
  Overall Score: 0.00
  Positive News Count: 0
  Negative News Count: 0
  Neutral News Count: 0
----------------------------------------
Sentiment Analysis for Britannia Industries Limited (BRITANNIA.NS):
  Overall Score: 0.00
  Positive News Count: 0
  Negative News Count: 0
  Neutral News Count: 0
----------------------------------------
Sentiment Analysis for Cipla Limited (CIPLA.NS):
  Overall Score: 0.00
  Positive News Count: 0
  Negative News Count: 0
  Neutral News Count: 0
----------------------------------------
Sentiment Analysis for Dr. Reddy's Laboratories Limited (DRREDDY.NS):
  Overall Score: 0.00
  Positive News Count: 0
  Negative News Count: 0
  Neutral News Count: 0
----------------------------------------
Sentiment Analysis for HDFC Bank Limited (HDFCBANK.NS):
  Overall Score: 0.00
  Positive News Count: 0
  Negative News Count: 0
  Neutral News Count: 0
----------------------------------------
Sentiment Ana